# Dependencias

In [ ]:
%pip install -e "..[dev]"

# Configuración


In [ ]:
from arqueo_pipeline.config import dias_atras, enviar_de_verdad, entorno
from arqueo_pipeline.extract import conectar, turnos_cerrados, ventas_de
from arqueo_pipeline.transform import mensaje, resumir

print("Días hacia atrás:", dias_atras())
print("Envía correo de verdad:", enviar_de_verdad())

# Extracción


In [ ]:
with conectar(entorno("DATABASE_URL")) as conn:
    turnos = turnos_cerrados(conn, dias_atras())
    ventas = ventas_de(conn, turnos["id_turno"].tolist() if not turnos.empty else [])

turnos.head()

# Dataframe

In [ ]:
resumen = resumir(turnos, ventas)
columnas = [
    "restaurante", "email", "cajero", "tickets",
    "esperado_efectivo", "efectivo_contado", "diferencia_efectivo",
    "esperado_digital", "digital_contado", "diferencia_digital",
]
resumen[columnas]

# Mensaje

In [ ]:
for _, fila in resumen.iterrows():
    asunto, cuerpo = mensaje(fila)
    print(fila["email"])
    print(asunto)
    print(cuerpo)
    print()